# From tissue to biological insight: the spatial transcriptomics pipeline

**Audience:** beginners who want to understand what happens before a Python spatial-transcriptomics analysis.

**Prerequisites:** basic ideas about DNA, RNA, genes, and gene expression. No coding is required.

**Purpose:** build a mental model of the complete workflow—from experimental design and tissue collection to a defensible biological conclusion.

By the end, you should be able to distinguish raw sequencing data, Space Ranger output, a processed `AnnData` object, and a biological result. You should also know where common technical and interpretive errors enter the pipeline.


## The complete journey

![Diagram of the full spatial transcriptomics workflow](../figures/00_pipeline_overview.svg)

Spatial transcriptomics is not one computational test. It is a chain in which each stage depends on the quality and assumptions of the previous stage.

| Stage | Main question | Main product |
|---|---|---|
| Experimental design | What biological comparison can answer the question? | Samples, groups, replicates, metadata |
| Tissue and spatial capture | Where did each RNA molecule originate? | Tissue image and spatially barcoded library |
| Sequencing | What nucleotide sequences were observed? | FASTQ files |
| Primary processing | Which gene and spot generated each molecule? | Gene-by-spot counts, coordinates, images |
| Python analysis | What patterns are present and are they spatial? | QC, clusters, markers, spatial statistics |
| Biological interpretation | What does the pattern mean and how certain are we? | A supported, limited biological conclusion |


## 1. Start with the biological question

The pipeline should begin before tissue enters a machine. A useful question names:

- the biological system: tissue, organ, disease, or developmental stage;
- the comparison: condition, region, treatment, genotype, or time point;
- the unit of replication: usually independent organisms or patients;
- the spatial outcome: domains, cell-type localization, boundaries, neighborhoods, or spatial gene programs.

**Example question**

> Which tissue regions and spatial gene-expression programs differ between untreated and treated tumors?

### Why design matters

Multiple sections from one patient provide technical or within-subject information, but they are not multiple independent patients. If treatment and processing batch are perfectly confounded, computation cannot reliably separate them. A beautiful tissue plot cannot repair an invalid design.


## 2. Prepare, stain, and image the tissue

A thin tissue section is placed on a capture area containing known spatial barcodes. The exact laboratory chemistry varies by platform, but the general idea is constant: preserve tissue position while measuring RNA.

Typical steps include:

1. collect and preserve tissue quickly enough to protect RNA and morphology;
2. cut a thin section and place it on the capture surface;
3. stain and image the section so morphology is visible;
4. permeabilize or otherwise release RNA;
5. capture RNA and attach a location-specific barcode;
6. convert captured RNA into a sequencing library.

### Biological risks introduced here

- degraded RNA lowers detected genes and molecules;
- folds, tears, necrosis, or poor staining affect local measurements;
- over- or under-permeabilization can reduce capture or blur spatial signal;
- tissue may extend beyond the capture area;
- adjacent sections are similar, but never perfectly identical.

The tissue image is not decoration. It is an independent morphological layer used to interpret molecular patterns.


## 3. Understand what one Visium spot represents

![Diagram showing that a Visium spot can contain multiple cells](../figures/00_spot_biology.svg)

In conventional spot-based Visium, one spot can receive RNA from multiple nearby cells. The resulting expression profile is therefore a local mixture.

This changes the language we should use:

- Prefer **spot**, **spatial domain**, or **region** when describing direct observations.
- Do not call a cluster a pure cell type without supporting evidence.
- Cell-type proportions may be estimated with deconvolution, but estimates depend on the reference and model.
- A marker gene can suggest a cell population or anatomical structure; it rarely proves identity alone.

Newer imaging-based and higher-resolution platforms can approach single-cell or subcellular resolution, but segmentation and molecule assignment introduce their own uncertainty.


## 4. What the sequencing machine produces

The sequencing instrument reads fragments from the prepared library. Its direct output is converted into **FASTQ files** containing:

- nucleotide sequences;
- a quality score for each called base;
- a spatial barcode identifying a capture location;
- a unique molecular identifier (**UMI**) that helps distinguish original molecules from PCR duplicates.

At this stage, the data do not yet form a clean gene-by-spot table. The sequencer does not report that “gene *Plp1* has 20 molecules in spot 17.” That interpretation requires primary processing.

### Key distinction

**Reads** are sequenced fragments. **UMI counts** are estimates of captured molecules after alignment, assignment, and duplicate collapsing. They are related, but not interchangeable.


## 5. Primary processing: from reads to spatial counts

For 10x Visium, **Space Ranger** commonly performs primary processing. Conceptually it:

1. checks barcode and read quality;
2. aligns gene-derived reads to a reference genome or transcriptome;
3. assigns aligned reads to genes;
4. uses spatial barcodes to assign molecules to spots;
5. collapses PCR duplicates using UMIs;
6. identifies which spots overlap tissue;
7. aligns the spot grid with the tissue image;
8. creates count matrices and quality summaries.

Important choices include the reference genome and annotation, tissue-detection accuracy, image alignment, and software version. These choices affect all downstream results.


## 6. What an analyst receives from Space Ranger

A typical output contains:

| Output | Meaning | Why it matters |
|---|---|---|
| Raw feature-barcode matrix | Counts for all barcodes | Useful for alternative tissue/spot calling |
| Filtered feature-barcode matrix | Counts for spots called as tissue | Common starting point for analysis |
| Tissue positions | Barcode-to-coordinate mapping | Connects expression to location |
| High/low-resolution images | Registered tissue morphology | Enables spatial plotting and pathology review |
| Scale factors | Relationship among image resolutions and coordinates | Keeps spots aligned with the image |
| Web summary and metrics | Sequencing, mapping, saturation, and tissue QC | First check of experiment quality |

This is much closer to the starting point of a Python analyst than FASTQ files. Re-running primary processing is important when references, annotations, or alignment choices must change; otherwise, validated Space Ranger output is often a reasonable starting point.


## 7. The four data layers used in Python

![Diagram of expression, coordinate, image, and metadata layers](../figures/00_data_layers.svg)

Packages such as AnnData, Scanpy, Squidpy, and SpatialData help keep these layers synchronized.

If expression rows and coordinate rows become reordered independently, gene expression can be drawn at the wrong tissue locations. Stable barcode identifiers and explicit validation are therefore essential.


## 8. Secondary analysis in Python

Secondary analysis turns count matrices and coordinates into interpretable patterns. The major stages are:

### 8.1 Inspect the complete object

Confirm the number of spots and genes, coordinate dimensions, sample metadata, count representation, image availability, and whether values have already been transformed.

### 8.2 Quality control

Evaluate counts per spot, genes detected per spot, mitochondrial fraction, tissue coverage, spatial distribution of QC values, image quality, and sample-level metrics.

### 8.3 Filter cautiously

Remove measurements supported as poor quality. A low-count region might be damaged tissue—or a real low-RNA anatomical structure. Always combine distributions with tissue location and morphology.

### 8.4 Normalize and transform

Sequencing depth varies among spots. Normalization makes spots more comparable, while a log-like transformation reduces domination by a few highly abundant genes. Preserve raw counts for methods that require them.

### 8.5 Select informative genes

Highly variable genes focus later steps on features that vary meaningfully. Selection can still capture technical variation, so QC and covariate awareness come first.

### 8.6 Reduce dimension and build an expression graph

PCA summarizes major expression programs. A nearest-neighbor graph then links spots with similar expression profiles. This graph is conceptually different from the spatial graph, which links spots because they are physically nearby.

### 8.7 Cluster and find markers

Clustering proposes molecularly similar groups; marker analysis describes genes enriched in each group. Biological annotation combines markers, spatial pattern, histology, and trusted references.

### 8.8 Add spatial analysis

Spatial methods ask questions that expression alone cannot:

- Are similar values located near one another? — spatial autocorrelation
- Which labeled regions touch more than expected? — neighborhood enrichment
- How do relationships change with distance? — co-occurrence
- Where are expression boundaries or spatial domains? — domain detection
- Which cell types may occupy each spot? — deconvolution

The method should follow the biological question, not the desire to produce another plot.


## 9. Two graphs that must not be confused

| Graph | Edges mean | Typical use |
|---|---|---|
| Expression-neighbor graph | Spots have similar molecular profiles | UMAP and Leiden clustering |
| Spatial-neighbor graph | Spots are physically close on the tissue | Moran's I and neighborhood enrichment |

A distant pair of spots can be expression neighbors if they represent the same tissue type. Two adjacent spots can be spatial neighbors while having very different expression because they lie on opposite sides of an anatomical boundary.


## 10. From a pattern to a biological claim

Suppose a gene has high Moran's I and forms a clear band across the tissue. A careful interpretation proceeds in layers:

1. **Observation:** expression is spatially autocorrelated.
2. **Description:** the high-expression band overlaps a particular anatomical region.
3. **Supporting evidence:** known markers, histology, or a reference atlas agree with the region.
4. **Alternative explanations:** library depth, tissue damage, cell-composition differences, or registration errors are considered.
5. **Claim:** state only what the design and evidence support.

Good language:

> Expression of gene X is enriched in a spatial domain overlapping the hippocampal region and is consistent with the reference annotation.

Overstated language:

> Gene X causes hippocampal organization.

Spatial association is not causation, and computational annotation is not experimental validation.


## 11. Validation and reproducibility

A defensible analysis records:

- sample identities, conditions, and biological replicates;
- tissue handling and image-review notes;
- reference genome and gene annotation;
- Space Ranger and Python package versions;
- QC plots and reasons for thresholds;
- whether each data layer contains raw counts, normalized values, or scaled values;
- parameters and random seeds;
- marker and annotation evidence;
- limitations and alternative explanations.

Biological validation may include independent tissue sections, additional subjects, pathology review, in situ hybridization, immunostaining, external datasets, or agreement with a trusted atlas.


## 12. Who usually owns each stage?

| Stage | Typical contributor | Analyst's responsibility |
|---|---|---|
| Study design | Biologist, clinician, statistician, bioinformatician | Check replication, metadata, contrasts, and confounding |
| Tissue preparation | Histology or spatial core laboratory | Understand protocols and record quality concerns |
| Sequencing | Sequencing core | Review run and library metrics |
| Primary processing | Core or bioinformatics team | Verify reference, parameters, alignment, and tissue detection |
| Secondary analysis | Bioinformatics/data science | Perform reproducible QC, modeling, visualization, and interpretation |
| Biological validation | Multidisciplinary team | Communicate uncertainty and propose follow-up tests |

You do not have to operate every machine to analyze the data well. You do need to understand what each stage contributes, what assumptions it makes, and what can go wrong.


## Exercise: trace a biological claim backward

Consider this result:

> A spatial domain near the tumor boundary shows increased interferon-response expression.

For each layer, write one question you would ask before trusting the claim:

1. **Biological design:** Were multiple independent tumors analyzed?
2. **Tissue:** Is the boundary intact and supported by histology?
3. **Sequencing/primary processing:** Are mapping and tissue-detection metrics acceptable?
4. **QC:** Could low RNA quality or depth create the pattern?
5. **Analysis:** How was the domain defined, and were multiple tests controlled?
6. **Interpretation:** Does the signal reflect expression within cells, a change in cell composition, or both?
7. **Validation:** What independent experiment or dataset could confirm it?

### Answer scaffold

A strong answer includes at least one alternative technical explanation, one alternative biological explanation, and one independent validation strategy. There is rarely a single plot that resolves all three.


## Common pitfalls

- Treating spots as single cells
- Calling clusters cell types from one marker gene
- Choosing QC thresholds without looking at tissue coordinates
- Using transformed expression as though it were raw counts
- Confusing the expression-neighbor graph with the spatial-neighbor graph
- Treating adjacent regions as proof of signaling
- Ignoring patient, sample, slide, or batch structure
- Reporting many sections from one subject as independent biological replicates
- Drawing population-level conclusions from one tissue section


## Knowledge check

You should now be able to answer:

1. What does the sequencing machine produce directly?
2. What does Space Ranger add to sequencing reads?
3. Why is a Visium spot not necessarily a cell?
4. What four data layers must remain aligned?
5. How does a spatial-neighbor graph differ from an expression-neighbor graph?
6. Why can a high Moran's I not establish biological mechanism?
7. Which part of the workflow is represented by notebook `01`?

**Connection to notebook 01:** the downloaded Squidpy object starts after primary processing and after several secondary-analysis steps. Notebook `01` uses it to teach spatial data structures and spatial statistics. A future raw-count notebook can focus specifically on QC, filtering, normalization, PCA, clustering, and marker analysis.
